# Validation diagnostics

Notebook solo diagnostico: non scrive risultati di regressione. Tutti gli output vanno nella cartella `diagnostics/`.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

SPATIAL_DIR = Path('../output/spatial_restricted')
REGRESSION_DIR = Path('../output/regressions/baseline_game_reserves')
DIAG_DIR = Path('../diagnostics')
DIAG_DIR.mkdir(parents=True, exist_ok=True)


## 1. Coverage variabili chiave


In [ ]:
key_vars = [
    'wave','survey_year','cluster_lat','cluster_lon','ward2022_name','district2022_name',
    'within_25km_game_reserve','distance_to_game_reserve_km',
    'maasai_pastoral_composite_proxy','pastoral_reference_50km',
    'log_cons_pc','log_cons_ae','cons_food_share_real'
]
rows = []
for path in SPATIAL_DIR.glob('analysis_*_spatial*.csv'):
    df = pd.read_csv(path, low_memory=False)
    for v in key_vars:
        if v in df.columns:
            rows.append({'file': path.name, 'variable': v, 'n': len(df), 'non_missing': df[v].notna().sum(), 'share_non_missing': df[v].notna().mean()})
coverage = pd.DataFrame(rows)
coverage.to_csv(DIAG_DIR / 'coverage_key_variables.csv', index=False)
coverage.head(30)


## 2. Support tables per wave e celle di trattamento


In [ ]:
rows = []
for path in SPATIAL_DIR.glob('analysis_*_spatial*.csv'):
    df = pd.read_csv(path, low_memory=False)
    needed = ['wave','within_25km_game_reserve','maasai_pastoral_composite_proxy','post_2009']
    if not set(needed).issubset(df.columns):
        continue
    d = df[needed].copy()
    d['cell'] = (d['within_25km_game_reserve'].fillna(0).astype(int).astype(str) + '_' +
                 d['maasai_pastoral_composite_proxy'].fillna(0).astype(int).astype(str) + '_' +
                 d['post_2009'].fillna(0).astype(int).astype(str))
    tab = d.groupby(['wave','cell'], dropna=False).size().reset_index(name='n')
    tab['file'] = path.name
    rows.append(tab)
support = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
support.to_csv(DIAG_DIR / 'support_cells_by_wave.csv', index=False)
support.head(30)


## 3. Baseline balance, solo descrittivo


In [ ]:
balance_vars = ['hh_size','head_age','share_female','n_children_u15','n_adults_15plus','log_cons_pc','log_cons_ae']
rows = []
for path in SPATIAL_DIR.glob('analysis_core_1_4_spatial*.csv'):
    df = pd.read_csv(path, low_memory=False)
    if 'wave' not in df.columns or 'within_25km_game_reserve' not in df.columns:
        continue
    base = df[pd.to_numeric(df['wave'], errors='coerce').eq(1)].copy()
    for v in balance_vars:
        if v not in base.columns:
            continue
        for group, g in base.groupby('within_25km_game_reserve'):
            x = pd.to_numeric(g[v], errors='coerce')
            rows.append({'file': path.name, 'variable': v, 'within_25km_game_reserve': group, 'n': x.notna().sum(), 'mean': x.mean(), 'sd': x.std()})
balance = pd.DataFrame(rows)
balance.to_csv(DIAG_DIR / 'baseline_balance_descriptive.csv', index=False)
balance.head(30)


## 4. Mappe rapide dei cluster, se coordinate disponibili


In [ ]:
for path in SPATIAL_DIR.glob('analysis_core_1_4_spatial*.csv'):
    df = pd.read_csv(path, low_memory=False)
    if {'cluster_lon','cluster_lat'}.issubset(df.columns):
        d = df[['cluster_lon','cluster_lat','within_25km_game_reserve']].dropna(subset=['cluster_lon','cluster_lat']).copy()
        plt.figure()
        plt.scatter(d['cluster_lon'], d['cluster_lat'], s=10, alpha=0.6)
        plt.xlabel('Longitude')
        plt.ylabel('Latitude')
        plt.title(f'Cluster coverage: {path.name}')
        plt.savefig(DIAG_DIR / f'map_clusters_{path.stem}.png', dpi=150, bbox_inches='tight')
        plt.show()
        break
